In [2]:
import os
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

In [11]:
DATA_PATH = '../mitbih_database/'

FS = 360
BEFORE = 100
AFTER = 180
TOLERANCE = 50

In [19]:
def load_annotations(path):
    return pd.read_csv(path, sep='\t')

def detect_r_peaks(signal):
    peaks, _ = find_peaks(
        signal,
        distance=int(0.45 * FS),
        prominence=0.3
    )

    return peaks

def find_annotation(peak, annotations, tolerance=50):
    annotation_samples = annotations['Sample #'].to_numpy()

    if len(annotation_samples) == 0:
        return None, None

    distances = np.abs(annotation_samples - peak)
    idx = np.argmin(distances)

    if distances[idx] <= tolerance:
        return (
            annotations.iloc[idx]['Type'],
            annotation_samples[idx]
        )

    return None, None

In [36]:
all_beats = []

record_files = [
    f for f in os.listdir(DATA_PATH)
    if f.endswith('_record.csv')
]

for record_file in record_files:
    record_id = record_file.replace('_record.csv', '')

    record_path = os.path.join(
        DATA_PATH,
        record_file
    )

    annotation_path = os.path.join(
        DATA_PATH,
        f'{record_id}_annotation.txt'
    )

    print(f'Processing record {record_id}...')
    
    ecg = pd.read_csv(record_path)
    
    if 'MLII' not in ecg.columns:
        print(f'{record_id}: brak MLII - pomijam')
        continue
    
    signal = ecg['MLII'].to_numpy()
    
    if os.path.exists(annotation_path):
        annotations = load_annotations(annotation_path)
    else:
        annotations = pd.DataFrame(
            columns=['Sample #', 'Type']
        )
        
    peaks = detect_r_peaks(signal)
    
    for peak in peaks:
        start = peak - BEFORE
        end = peak + AFTER
        
        if start < 0 or end > len(signal):
            continue
        
        beat = signal[start:end]
        
        beat_type, annotation_sample = find_annotation(
            peak,
            annotations,
            tolerance=TOLERANCE
        )
        
        row = {
            'record_id': record_id,
            'r_peak': peak,
            'annotation_sample': annotation_sample,
            'type': beat_type,
            'dist_from_annotation': abs(annotation_sample - peak) if annotation_sample is not None else np.nan
        }
        
        for i, value in enumerate(beat):
            row[f'x_{i}'] = value
        
        all_beats.append(row)

Processing record 100...
Processing record 101...
Processing record 102...
102: brak MLII - pomijam
Processing record 103...
Processing record 104...
104: brak MLII - pomijam
Processing record 105...
Processing record 106...
Processing record 107...
Processing record 108...
Processing record 109...
Processing record 111...
Processing record 112...
Processing record 113...
Processing record 114...
Processing record 115...
Processing record 116...
Processing record 117...
Processing record 118...
Processing record 119...
Processing record 121...
Processing record 122...
Processing record 123...
Processing record 124...
Processing record 200...
Processing record 201...
Processing record 202...
Processing record 203...
Processing record 205...
Processing record 207...
Processing record 208...
Processing record 209...
Processing record 210...
Processing record 212...
Processing record 213...
Processing record 214...
Processing record 215...
Processing record 217...
Processing record 219...


In [37]:
beats_df = pd.DataFrame(all_beats)

In [38]:
beats_df.head()

,record_id,r_peak,annotation_sample,type,dist_from_annotation,x_0,x_1,x_2,x_3,x_4,...,x_270,x_271,x_272,x_273,x_274,x_275,x_276,x_277,x_278,x_279
0,100,370,370.0,N,0.0,-0.315,-0.300,-0.305,-0.310,-0.320,...,-0.345,-0.335,-0.33,-0.330,-0.345,-0.345,-0.345,-0.320,-0.330,-0.335
1,100,663,662.0,N,1.0,-0.345,-0.345,-0.330,-0.330,-0.335,...,-0.380,-0.370,-0.36,-0.365,-0.360,-0.365,-0.380,-0.370,-0.365,-0.350
2,100,947,946.0,N,1.0,-0.345,-0.345,-0.340,-0.355,-0.365,...,-0.335,-0.335,-0.33,-0.335,-0.340,-0.335,-0.320,-0.340,-0.340,-0.350
3,100,1231,1231.0,N,0.0,-0.350,-0.350,-0.350,-0.340,-0.340,...,-0.335,-0.340,-0.33,-0.320,-0.315,-0.325,-0.335,-0.355,-0.345,-0.340
4,100,1515,1515.0,N,0.0,-0.330,-0.315,-0.315,-0.330,-0.330,...,-0.320,-0.310,-0.31,-0.310,-0.330,-0.335,-0.330,-0.320,-0.325,-0.325


In [39]:
beats_df.shape

(103646, 285)

In [40]:
beats_df['type'].value_counts(dropna=False)

type
N      73295
R       7255
L       6520
NaN     4723
V       4205
/       3612
A       2215
F        781
f        259
j        228
!        190
E        102
a         98
J         73
|         24
~         18
+         16
e         16
Q          9
x          3
S          2
]          1
"          1
Name: count, dtype: int64

### Typy uderzeń EKG wykorzystane w eksperymencie

| Symbol | Typ uderzenia | Klasa w modelu |
|:------:|---------------|:--------------:|
| `N` | Normal beat – normalne uderzenie serca | **Normal** |
| `L` | Left bundle branch block beat – uderzenie z blokiem lewej odnogi pęczka Hisa | **Anomaly** |
| `R` | Right bundle branch block beat – uderzenie z blokiem prawej odnogi pęczka Hisa | **Anomaly** |
| `A` | Atrial premature beat – przedwczesne uderzenie przedsionkowe | **Anomaly** |
| `a` | Aberrated atrial premature beat – aberracyjne przedwczesne uderzenie przedsionkowe | **Anomaly** |
| `J` | Nodal (junctional) premature beat – przedwczesne uderzenie węzłowe | **Anomaly** |
| `S` | Supraventricular premature beat – przedwczesne uderzenie nadkomorowe | **Anomaly** |
| `V` | Premature ventricular contraction – przedwczesne pobudzenie komorowe (PVC) | **Anomaly** |
| `E` | Ventricular escape beat – komorowe pobudzenie zastępcze | **Anomaly** |
| `F` | Fusion of ventricular and normal beat – uderzenie fuzyjne komorowe i prawidłowe | **Anomaly** |

In [41]:
normal_types = ['N']
anomaly_types = ['L', 'R', 'A', 'a', 'J', 'S', 'V', 'E', 'F']

valid_types = normal_types + anomaly_types

model_df = beats_df[
    beats_df['type'].isin(valid_types)
].copy()

model_df['anomaly'] = np.where(
    model_df['type'] == 'N',
    0,
    1
)

In [42]:
model_df.head()

,record_id,r_peak,annotation_sample,type,dist_from_annotation,x_0,x_1,x_2,x_3,x_4,...,x_271,x_272,x_273,x_274,x_275,x_276,x_277,x_278,x_279,anomaly
0,100,370,370.0,N,0.0,-0.315,-0.300,-0.305,-0.310,-0.320,...,-0.335,-0.33,-0.330,-0.345,-0.345,-0.345,-0.320,-0.330,-0.335,0
1,100,663,662.0,N,1.0,-0.345,-0.345,-0.330,-0.330,-0.335,...,-0.370,-0.36,-0.365,-0.360,-0.365,-0.380,-0.370,-0.365,-0.350,0
2,100,947,946.0,N,1.0,-0.345,-0.345,-0.340,-0.355,-0.365,...,-0.335,-0.33,-0.335,-0.340,-0.335,-0.320,-0.340,-0.340,-0.350,0
3,100,1231,1231.0,N,0.0,-0.350,-0.350,-0.350,-0.340,-0.340,...,-0.340,-0.33,-0.320,-0.315,-0.325,-0.335,-0.355,-0.345,-0.340,0
4,100,1515,1515.0,N,0.0,-0.330,-0.315,-0.315,-0.330,-0.330,...,-0.310,-0.31,-0.310,-0.330,-0.335,-0.330,-0.320,-0.325,-0.325,0


In [43]:
model_df['anomaly'].value_counts()

anomaly
0    73295
1    21251
Name: count, dtype: int64

In [44]:
model_df['type'].value_counts()

type
N    73295
R     7255
L     6520
V     4205
A     2215
F      781
E      102
a       98
J       73
S        2
Name: count, dtype: int64

Do modelu OCSVM - całość danych. Do modeli klasyfikacyjnych wykorzystane będą tylko najczęstsze typy anomalii (ze względu na jakość uczenia), tzn. N, R, L, V, A.

In [45]:
output_path = 'clean_data.csv'

model_df.to_csv(output_path)